# Small Reasoning LLM Lab — Training Notebook

This notebook is an **orchestration interface only**. All model, training, and
evaluation code lives in the repository. The notebook:

1. Installs dependencies
2. Clones the repository
3. Detects the GPU
4. **Benchmarks actual hardware performance** (steps/sec, tokens/sec, memory)
5. Configures the experiment
6. Generates the dataset
7. Trains the model from random initialisation
8. Evaluates on unseen problems
9. Exports artifacts

**Before running:** Set `Runtime → Change runtime type → T4 GPU`

**Model:** 8m preset = 7,949,824 trainable parameters (weight-tied, d256/8L/ff1152)

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install tokenizers>=0.15.0 tqdm pyyaml rich -q
print('Dependencies installed.')

## 2. Clone / Mount the Repository

In [ ]:
import os, sys

# Option A: Clone from GitHub
REPO_URL  = 'https://github.com/YOUR_USERNAME/small-llm-lab.git'  # <-- update
REPO_NAME = 'small-llm-lab'

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
else:
    print(f'{REPO_NAME} already cloned.')

os.chdir(REPO_NAME)
sys.path.insert(0, '.')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/small-llm-lab')
# sys.path.insert(0, '.')

## 3. Detect GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU:           {gpu_name}')
    print(f'GPU memory:    {gpu_mem:.1f} GB')
    print(f'CUDA version:  {torch.version.cuda}')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU detected. Training will be very slow.')
    print('Go to Runtime -> Change runtime type -> T4 GPU')
    DEVICE = 'cpu'

print(f'PyTorch:       {torch.__version__}')
print(f'Device:        {DEVICE}')

## 4. Hardware Benchmark

Run this before training. It measures the **actual** steps/sec and tokens/sec
on your specific GPU using the real 8m model config and training batch size.
This gives a hardware-specific runtime estimate rather than a CPU-scaled guess.

Results will vary by GPU model, CUDA version, and available memory.

In [ ]:
import time, math
import torch
import torch.nn as nn
from model.config import get_8m_config
from model.transformer import SmallTransformer

# ── Benchmark config (matches colab_small training config) ────────────────
BENCH_BATCH     = 64     # matches train_config.batch_size
BENCH_SEQ       = 200    # realistic sequence length for reasoning format
BENCH_WARMUP    = 10     # warmup steps (not timed)
BENCH_STEPS     = 50     # timed steps

# ── Build model ───────────────────────────────────────────────────────────
cfg = get_8m_config()
cfg.vocab_size  = 4096
cfg.max_seq_len = 256
bench_model = SmallTransformer(cfg).to(DEVICE)
bench_model.train()
bench_opt = torch.optim.AdamW(bench_model.parameters(), lr=3e-4)
n_params = bench_model.count_parameters()
print(f'Benchmark model: {n_params:,} trainable parameters')

# ── Dummy batch ───────────────────────────────────────────────────────────
xb = torch.randint(1, 4096, (BENCH_BATCH, BENCH_SEQ), device=DEVICE)
yb = torch.randint(1, 4096, (BENCH_BATCH, BENCH_SEQ), device=DEVICE)

# ── Warmup ────────────────────────────────────────────────────────────────
for _ in range(BENCH_WARMUP):
    loss, _ = bench_model(xb, yb)
    bench_opt.zero_grad()
    loss.backward()
    bench_opt.step()

if DEVICE == 'cuda':
    torch.cuda.synchronize()

# ── Timed run ─────────────────────────────────────────────────────────────
t0 = time.perf_counter()
for _ in range(BENCH_STEPS):
    loss, _ = bench_model(xb, yb)
    bench_opt.zero_grad()
    loss.backward()
    bench_opt.step()

if DEVICE == 'cuda':
    torch.cuda.synchronize()
t1 = time.perf_counter()

# ── Peak GPU memory ───────────────────────────────────────────────────────
if DEVICE == 'cuda':
    peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
else:
    peak_mem_gb = 0.0
    total_mem_gb = 0.0

# ── Metrics ───────────────────────────────────────────────────────────────
elapsed       = t1 - t0
steps_per_sec = BENCH_STEPS / elapsed
tokens_per_sec = steps_per_sec * BENCH_BATCH * BENCH_SEQ

# ── colab_small runtime estimate ──────────────────────────────────────────
# 20k examples x ~80 avg tokens x 30 epochs = ~48M tokens total
# With grad_accum=2, effective batch is 128, so steps = 48M / (128 * avg_tokens) * grad_accum
COLAB_TOTAL_TOKENS = 20_000 * 80 * 30   # rough: 48M
estimated_seconds  = COLAB_TOTAL_TOKENS / tokens_per_sec

print()
print('=== Hardware Benchmark Results ===')
print(f'  GPU:              {torch.cuda.get_device_name(0) if DEVICE=="cuda" else "CPU"}')
print(f'  Steps/sec:        {steps_per_sec:.1f}  (batch={BENCH_BATCH}, seq={BENCH_SEQ})')
print(f'  Tokens/sec:       {tokens_per_sec:,.0f}')
print(f'  Peak GPU memory:  {peak_mem_gb:.2f} GB / {total_mem_gb:.1f} GB total')
print()
print(f'  colab_small estimated training time: {estimated_seconds/60:.0f} minutes')
print(f'  (based on ~48M tokens, actual time depends on seq lengths and batch packing)')
print()
print('NOTE: This estimate is hardware-specific and measured on this exact GPU.')
print('      It will differ on A100, V100, or other GPU types.')

# Clean up benchmark model
del bench_model, bench_opt, xb, yb
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## 5. Configure the Experiment

In [ ]:
from training.config import get_train_config_by_name

# Presets:
#   'debug'        -- tiny model, ~30 sec, for pipeline testing
#   'colab_small'  -- 7.95M model, 20k examples, see benchmark above for time
#   'colab_medium' -- 7.95M model, 50k examples

PRESET = 'colab_small'   # <-- change this

train_config = get_train_config_by_name(PRESET)

# Optional overrides:
# train_config.max_epochs    = 20
# train_config.learning_rate = 1e-4
# train_config.use_reasoning = False   # ablation: direct format

print(f'Experiment:  {train_config.experiment_name}')
print(f'Model:       {train_config.model_size} preset (7,949,824 params, weight-tied)')
print(f'N train:     {train_config.n_train:,}')
print(f'N val:       {train_config.n_val:,}')
print(f'Reasoning:   {train_config.use_reasoning}')
print(f'Output dir:  {train_config.output_dir}')

## 6. Generate Dataset (deduped, zero cross-split overlap)

In [ ]:
from data.generators.arithmetic import ArithmeticGenerator, build_corpus

generator = ArithmeticGenerator(
    seed=train_config.data_seed,
    difficulty=train_config.difficulty,
)

# generate_all_splits() guarantees zero exact problem overlap across splits
train_examples, val_examples, test_examples = generator.generate_all_splits(
    n_train=train_config.n_train,
    n_val=train_config.n_val,
    n_test=train_config.n_test,
)

# Verify
tp = set(e.problem for e in train_examples)
vp = set(e.problem for e in val_examples)
sp = set(e.problem for e in test_examples)
assert len(tp & vp) == 0 and len(tp & sp) == 0 and len(vp & sp) == 0

print(f'Train: {len(train_examples):,} unique examples')
print(f'Val:   {len(val_examples):,} unique examples')
print(f'Test:  {len(test_examples):,} unique examples')
print(f'Split overlap: train/val=0, train/test=0, val/test=0 (verified)')
print()
print('--- Sample (reasoning format) ---')
print(train_examples[0].to_reasoning_text())

## 7. Train the Model

In [ ]:
from training.train import train

# Runs the full pipeline:
#   trains BPE tokenizer -> builds DataLoaders ->
#   initialises model from random weights ->
#   training loop with validation + reasoning evals ->
#   saves best checkpoint ->
#   final benchmark + generalization test
#
# Checkpoints saved:
#   init.pt        -- before any training
#   early.pt       -- after ~10% of training
#   mid.pt         -- after ~50% of training
#   step_XXXXXXX.pt -- periodic checkpoints
#   best.pt        -- best validation loss
#
# If the session disconnects, resume by setting:
#   train_config.resume_from = 'experiments/results/.../checkpoints/best.pt'

results = train(train_config)

print('\n=== Final Results ===')
for k, v in results.items():
    if not isinstance(v, dict):
        print(f'  {k}: {v}')

## 8. Evaluate on Unseen Problems

In [ ]:
import os, torch, json
from model.config import ModelConfig
from model.transformer import SmallTransformer
from model.tokenizer import BPETokenizer
from training.checkpoint import load_checkpoint
from evaluation.benchmark import run_benchmark, run_generalization_benchmark

device = 'cuda' if torch.cuda.is_available() else 'cpu'

ck_path = os.path.join(train_config.checkpoint_dir, 'best.pt')
ck      = torch.load(ck_path, map_location=device)

model_config = ModelConfig.from_dict(ck['model_config'])
model        = SmallTransformer(model_config)
model.load_state_dict(ck['model_state'])
model.to(device).eval()
tokenizer = BPETokenizer.load(train_config.tokenizer_dir)

print(f'Checkpoint: step {ck["global_step"]}')
print(f'Parameters: {model.count_parameters():,}')

bench = run_benchmark(
    model=model, tokenizer=tokenizer, generator=generator,
    n_problems=500, split='test',
    max_new_tokens=64, temperature=0.0,
    use_reasoning=train_config.use_reasoning,
    max_samples_to_save=20, device=device,
)
print('\n' + bench.summary_str())

In [ ]:
# 5-level generalization benchmark
# Passes all known problems as exclusion set to guarantee zero overlap
all_known = set(e.problem for e in train_examples + val_examples + test_examples)
gen_results = run_generalization_benchmark(
    model=model, tokenizer=tokenizer, generator=generator,
    n_per_level=200, device=device,
    use_reasoning=train_config.use_reasoning,
    exclude_problems=all_known,
)
print('Generalization accuracy by level:')
for level, r in gen_results.items():
    print(f'  Level {level}: {r["accuracy"]:.1%} ({r["correct"]}/{r["total"]})')

## 9. Inspect Sample Outputs

In [ ]:
for sample in bench.samples[:10]:
    status = 'CORRECT' if sample['correct'] else 'WRONG'
    print(f"[{status}] [{sample['family']}] {sample['problem']}")
    print(f"  Expected: {sample['expected']}")
    print(f"  Got:      {sample['predicted']}")
    print()

## 10. Progression Check (init vs early vs mid vs best)

In [ ]:
# Evaluate each progression checkpoint on the same frozen test problems
# to verify the model actually learned over time.
ck_dir = train_config.checkpoint_dir
labels = ['init', 'early', 'mid', 'best']

print(f'{"Checkpoint":<12} {"Step":>8} {"Test Acc":>10}')
print('-' * 35)

for label in labels:
    ck_file = os.path.join(ck_dir, f'{label}.pt')
    if not os.path.exists(ck_file):
        print(f'{label:<12} not found')
        continue
    ck_i = torch.load(ck_file, map_location=device)
    m_i  = SmallTransformer(ModelConfig.from_dict(ck_i['model_config']))
    m_i.load_state_dict(ck_i['model_state'])
    m_i.to(device).eval()
    b_i = run_benchmark(
        model=m_i, tokenizer=tokenizer, generator=generator,
        n_problems=200, split='test',
        max_new_tokens=64, temperature=0.0,
        use_reasoning=train_config.use_reasoning,
        max_samples_to_save=0, device=device,
    )
    step = ck_i.get('global_step', '?')
    print(f'{label:<12} {step:>8} {b_i.accuracy:>10.1%}')
    del m_i

## 11. Export Artifacts

In [ ]:
import shutil
output_dir   = train_config.output_dir
archive_name = output_dir.replace('/', '_').replace('\\', '_')
shutil.make_archive(archive_name, 'zip', output_dir)
print(f'Archived: {archive_name}.zip')

try:
    from google.colab import files
    files.download(f'{archive_name}.zip')
except ImportError:
    print(f'Not in Colab. Archive saved as {archive_name}.zip')

## Notes

- The GPU benchmark in Section 4 measures actual hardware performance and gives
  a training time estimate specific to this session's GPU. Do not rely on CPU-scaled estimates.
- `init.pt`, `early.pt`, `mid.pt`, and `best.pt` are saved automatically.
  Use Section 10 to verify learning progression before drawing conclusions.
- If a session disconnects, resume: `train_config.resume_from = 'path/to/checkpoint'`
- The verifier is programmatic — no LLM is used as a judge.
- Low training loss does NOT mean the model solved problems. Check test accuracy.
- Checkpoint frequency: every `save_every_n_steps` steps (default 1000).
